<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/seq2one/stage_07_07_tcn_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_06 -  Modelo GRU (many-to-one)**

El **GRU (Gated Recurrent Unit)** es una red neuronal recurrente similar al LSTM, pero con una arquitectura **más simple** y **menos parámetros**.

Al igual que el LSTM, el GRU modela explícitamente la **dinámica temporal** de la secuencia histórica, procesando los datos minuto a minuto y resumiendo la
información pasada en un estado oculto.

En configuración **many-to-one**, el modelo recibe una secuencia histórica
(60 x 20) y produce un **único valor escalar futuro**.

El GRU suele ser **más estable y eficiente** que el LSTM, y en muchos problemas
intradiarios logra desempeño comparable con menor riesgo de sobreajuste.



# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [5]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [6]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows_seq2one/train_delta60_ws60.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows_seq2one/test_delta60_ws60.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows_seq2one/valid_delta60_ws60.npz"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows_seq2one/train_delta90_ws60.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows_seq2one/test_delta90_ws60.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows_seq2one/valid_delta90_ws60.npz"))

#ESCALADOR GLOBAL
IN_SCALER = Path(os.environ.get("IN_SCALER", "data/scaled/scaler.joblib"))



In [7]:
#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
#IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
#IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [8]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [9]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z

IN_SCALER = DRIVE_DIR / IN_SCALER

#IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
#IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [10]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [11]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [12]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [13]:
# Construye un StageConfig leyendo ambos reports.
SUMMARY = """
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )
"""

In [14]:
#states_h60 = load_state_from_reports(horizon=60)
#states_h60

In [15]:
#states_h90 = load_state_from_reports(horizon=90)
#states_h90


## **4. Importar métricas comunes desde .py**

In [16]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [17]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **5. Carga de data windows**

In [18]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [19]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [20]:
# Cargar scaler
#scaler = joblib.load("/content/drive/MyDrive/neural_profit/data/scaled/scaler.joblib")

In [21]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [22]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


In [23]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (330144, 1200) (330144,)
H60 Valid: (70952, 1200) (70952,)
H60 Test : (70590, 1200) (70590,)
H90 Train: (330144, 1200) (330144,)
H90 Valid: (70952, 1200) (70952,)
H90 Test : (70590, 1200) (70590,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [24]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [25]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [26]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [27]:
summary = run_sanity_checks_all_horizons_seq2one(bundle_60, bundle_90)

#summary["h60"]["train"]

[sanity_check_seq2one] train_h60 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=57.959399
[sanity_check_seq2one] valid_h60 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=92.974502
[sanity_check_seq2one] test_h60 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=54.347220
OK h60 (h=60)
[sanity_check_seq2one] train_h90 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=70.925909
[sanity_check_seq2one] valid_h90 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=114.837946
[sanity_check_seq2one] test_h90 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=67.180748
OK h90 (h=90)


# **DEFINICIÓN DE MODELO**

## **7. Definición del modelo — placeholder**

### **7.1. Modelo GRU - many to one**

**Idea básica**

El **GRU (Gated Recurrent Unit)** es una red neuronal recurrente diseñada para
modelar **dependencias temporales** en secuencias, de forma similar al LSTM,
pero con una arquitectura **más compacta** y un menor número de parámetros.

A diferencia del MLP, el GRU **preserva explícitamente la estructura temporal**
de la ventana histórica y procesa la información **paso a paso en el tiempo**.
En comparación con el LSTM, el GRU reemplaza el mecanismo de tres compuertas
por dos, simplificando la dinámica interna.

En configuración **many-to-one**, el modelo utiliza el último estado oculto
como resumen de toda la secuencia para predecir un valor escalar futuro.

Formalmente, el GRU se define mediante:

$$
z_t = \sigma(W_z x_t + U_z h_{t-1})
$$

$$
r_t = \sigma(W_r x_t + U_r h_{t-1})
$$

$$
\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}))
$$

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

$$
\hat{y}_t = W_o h_T + b_o
$$

donde:
- $x_t \in \mathbb{R}^{20}$ es el vector de features en el minuto $t$,
- $h_t$ es el estado oculto del GRU,
- $z_t$ es la compuerta de actualización,
- $r_t$ es la compuerta de reinicio,
- $h_T$ resume toda la ventana histórica (por ejemplo, 60 minutos),
- $W_o, b_o$ son los parámetros de la capa de salida.

---

**Regularización (GRU)**

**Riesgo:** Medio, generalmente menor que en LSTM debido a su menor capacidad.

La regularización se controla principalmente de forma **estructural**:

- **Early stopping:** mecanismo principal para evitar sobreajuste.
- **Dimensión del estado oculto:** mantener un tamaño moderado.
- **Número de capas limitado:** 1 (máximo 2).
- **Dropout (opcional):**
  - Aplicado solo entre capas (si `num_layers > 1`),
  - No dentro de la recurrencia.

En muchos casos, el GRU requiere **menos regularización explícita** que el LSTM
para lograr una generalización comparable.

---

**Por qué el GRU es relevante en este proyecto**

- Entrada **secuencial explícita**: 60 × 20.
- Capacidad para capturar:
  - dependencias temporales de corto y mediano plazo,
  - dinámica intradía sin aplanar la información.
- Modelo:
  - más simple que LSTM,
  - más eficiente en memoria y tiempo,
  - potencialmente más estable en entrenamiento.

El GRU actúa como una **alternativa recurrente robusta** frente al LSTM,
permitiendo evaluar si una arquitectura más simple logra un mejor compromiso
entre desempeño y estabilidad.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: GRU many-to-one
- Número de capas: 1
- Dimensión del estado oculto: moderada (por ejemplo, 64-256)
- Dropout: desactivado inicialmente
- Optimización: Adam
- Early stopping: activado
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


### **7.2. Imports (PyTorch) + semillas**

In [29]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### **7.3. Utilidad: reshape de X desde (n, 1200) a (n, 60, 20)**

In [30]:
def reshape_X_flat_to_seq(X_flat: np.ndarray, *, seq_len: int = 60, n_features: int = 20) -> np.ndarray:
    """
    Convierte X de (n, flat_dim) a (n, seq_len, n_features).
    Espera flat_dim = seq_len * n_features.
    """
    X_flat = np.asarray(X_flat, dtype=np.float32)
    if X_flat.ndim != 2:
        raise ValueError(f"Se espera X 2D (n, flat_dim). Recibido: {X_flat.shape}")

    n, flat_dim = X_flat.shape
    expected = seq_len * n_features
    if flat_dim != expected:
        raise ValueError(f"flat_dim={flat_dim} != seq_len*n_features={expected} ({seq_len}*{n_features})")

    return X_flat.reshape(n, seq_len, n_features)


### **7.4. DataLoaders desde bundle (con reshape interno)**

In [31]:
def make_lstm_loaders_from_bundle(
    bundle: dict,
    *,
    seq_len: int = 60,
    n_features: int = 20,
    batch_size: int = 16384,
    num_workers: int = 0,
) -> dict:
    """
    Crea loaders train/valid/test para LSTM many-to-one.
    - X: (n, 1200) -> (n, 60, 20)
    - y: (n,) -> (n, 1)
    """
    loaders = {}
    for split in ["train", "valid", "test"]:
        X_flat = bundle[split]["X"]
        y = bundle[split]["y"]

        X = reshape_X_flat_to_seq(X_flat, seq_len=seq_len, n_features=n_features)           # (n,60,20)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)                                   # (n,1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders


In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders_gru_60 = make_lstm_loaders_from_bundle(bundle_60, seq_len=60, n_features=20, batch_size=16384)
loaders_gru_90 = make_lstm_loaders_from_bundle(bundle_90, seq_len=60, n_features=20, batch_size=16384)


### **7.5. Modelo GRU many-to-one**

In [33]:
class GRUSeq2One(nn.Module):
    def __init__(self, *, n_features: int = 20, hidden_size: int = 256, num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        _, h_n = self.gru(x)     # h_n: (num_layers, batch, hidden)
        last_h = h_n[-1]         # (batch, hidden)
        return self.head(last_h) # (batch, 1)

### **7.6. Train: early stopping + gradient clipping + scheduler**



In [34]:
@torch.no_grad()
def eval_mse(model: nn.Module, loader, device: torch.device) -> float:
    model.eval()
    mse_sum, n = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)


def train_gru_tuned(
    loaders: dict,
    *,
    n_features: int = 20,
    hidden_size: int = 256,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 5e-4,
    weight_decay: float = 1e-4,
    max_epochs: int = 40,
    patience: int = 6,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,
    device: torch.device,
) -> tuple[nn.Module, dict]:
    model = GRUSeq2One(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=2, min_lr=1e-5
        )

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    history = {"best_valid_mse": None, "epochs_ran": 0, "final_lr": None}

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(clip_grad_norm))

            opt.step()

        valid_mse = eval_mse(model, loaders["valid"], device)

        if scheduler is not None:
            scheduler.step(valid_mse)

        current_lr = opt.param_groups[0]["lr"]
        print(
            f"epoch={epoch:02d} | valid_mse={valid_mse:.6f} | lr={current_lr:.2e} "
            f"| hs={hidden_size} | wd={weight_decay:.1e} | L={num_layers} | do={dropout:.2f}"
        )

        history["epochs_ran"] = epoch
        history["final_lr"] = current_lr

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    history["best_valid_mse"] = float(best_valid)
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history



### **7.7. Predicción GRU (VALID/TEST) desde X_flat**


In [35]:
@torch.no_grad()
def predict_gru(
    model: nn.Module,
    X_flat: np.ndarray,
    *,
    device: torch.device,
    seq_len: int = 60,
    n_features: int = 20,
    batch_size: int = 2048,
) -> np.ndarray:
    model.eval()
    X_seq = reshape_X_flat_to_seq(X_flat, seq_len=seq_len, n_features=n_features)  # (n,60,20)
    n = X_seq.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_seq[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)
        preds.append(yb.detach().cpu().numpy())
        del xb, yb

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(preds, axis=0)



### **7.8. Runner GRU: grid pequeño por horizonte + métricas + tabla**


In [36]:
import pandas as pd
import torch

def run_gru_grid_for_bundle(
    bundle: dict,
    *,
    horizon: int,
    device: torch.device,
    seq_len: int = 60,
    n_features: int = 20,
    batch_size_train: int = 4096,
    batch_size_pred: int = 2048,
    grid: list[dict],
) -> pd.DataFrame:
    """
    Corre una grilla de configs GRU para un horizonte y devuelve tabla de resultados (valid/test).

    Requiere que existan:
      - make_lstm_loaders_from_bundle(bundle, ...)
      - train_gru_tuned(loaders, ...)
      - predict_gru(model, X_flat, ...)
      - compute_seq2one_metrics(y_true, y_pred, compute_r2=True)
      - metrics_to_df(metrics, model=..., split=..., horizon=...)
    """
    loaders = make_lstm_loaders_from_bundle(
        bundle,
        seq_len=seq_len,
        n_features=n_features,
        batch_size=batch_size_train,
    )

    y_valid = bundle["valid"]["y"]
    y_test  = bundle["test"]["y"]

    rows = []

    for i, cfg in enumerate(grid, start=1):
        print(f"\n--- GRU GRID {i}/{len(grid)} | h={horizon} | cfg={cfg} ---")

        model, hist = train_gru_tuned(
            loaders,
            n_features=n_features,
            hidden_size=cfg.get("hidden_size", 128),
            num_layers=cfg.get("num_layers", 1),
            dropout=cfg.get("dropout", 0.0),
            lr=cfg.get("lr", 1e-3),
            weight_decay=cfg.get("weight_decay", 1e-4),
            max_epochs=cfg.get("max_epochs", 40),
            patience=cfg.get("patience", 6),
            clip_grad_norm=cfg.get("clip_grad_norm", 1.0),
            use_scheduler=cfg.get("use_scheduler", True),
            device=device,
        )

        # Predicciones
        y_pred_valid = predict_gru(
            model,
            bundle["valid"]["X"],
            device=device,
            seq_len=seq_len,
            n_features=n_features,
            batch_size=batch_size_pred,
        )
        y_pred_test = predict_gru(
            model,
            bundle["test"]["X"],
            device=device,
            seq_len=seq_len,
            n_features=n_features,
            batch_size=batch_size_pred,
        )

        # Métricas
        m_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)
        m_test  = compute_seq2one_metrics(y_test,  y_pred_test,  compute_r2=True)

        # Filas
        df_v = metrics_to_df(m_valid, model="gru", split="valid", horizon=horizon)
        df_t = metrics_to_df(m_test,  model="gru", split="test",  horizon=horizon)

        # Trazabilidad: hiperparámetros + entrenamiento
        for k, v in cfg.items():
            df_v[k] = v
            df_t[k] = v

        df_v["best_valid_mse"] = hist["best_valid_mse"]
        df_t["best_valid_mse"] = hist["best_valid_mse"]
        df_v["epochs_ran"] = hist["epochs_ran"]
        df_t["epochs_ran"] = hist["epochs_ran"]
        df_v["final_lr"] = hist["final_lr"]
        df_t["final_lr"] = hist["final_lr"]

        rows.append(df_v)
        rows.append(df_t)

        # ---- liberar GPU entre corridas ----
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["split", "horizon_min", "RMSE"]).reset_index(drop=True)
    return out


### **7.10. Grid y ejecución**


In [38]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])


In [37]:
grid = [
    {"hidden_size": 128, "lr": 1e-3, "weight_decay": 1e-4},
    {"hidden_size": 128, "lr": 5e-4, "weight_decay": 1e-4},
    {"hidden_size": 128, "lr": 2e-4, "weight_decay": 1e-4},
    {"hidden_size": 256, "lr": 1e-3, "weight_decay": 1e-4},
    {"hidden_size": 256, "lr": 5e-4, "weight_decay": 1e-4},
    {"hidden_size": 256, "lr": 2e-4, "weight_decay": 1e-4},
]


In [40]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
df_gru_grid_60 = run_gru_grid_for_bundle(
    bundle_60,
    horizon=60,
    device=device,
    seq_len=60,
    n_features=20,
    batch_size_train=4096,
    batch_size_pred=2048,
    grid=grid,
)


--- GRU GRID 1/6 | h=60 | cfg={'hidden_size': 128, 'lr': 0.001, 'weight_decay': 0.0001} ---
epoch=01 | valid_mse=8646.281690 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=02 | valid_mse=8646.864373 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=03 | valid_mse=8666.925111 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=04 | valid_mse=8675.213546 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=05 | valid_mse=8672.492826 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=06 | valid_mse=8676.533819 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=07 | valid_mse=8676.538265 | lr=2.50e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
Early stopping (patience=6). Best valid_mse=8646.281690

--- GRU GRID 2/6 | h=60 | cfg={'hidden_size': 128, 'lr': 0.0005, 'weight_decay': 0.0001} ---
epoch=01 | valid_mse=8640.623880 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=02 | valid_mse=8654.131187 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | 

In [42]:
df_gru_grid_90 = run_gru_grid_for_bundle(
    bundle_90,
    horizon=90,
    device=device,
    seq_len=60,
    n_features=20,
    batch_size_train=4096,
    batch_size_pred=2048,
    grid=grid,
)


--- GRU GRID 1/6 | h=90 | cfg={'hidden_size': 128, 'lr': 0.001, 'weight_decay': 0.0001} ---
epoch=01 | valid_mse=13180.107791 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=02 | valid_mse=13206.213581 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=03 | valid_mse=13212.807870 | lr=1.00e-03 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=04 | valid_mse=13232.988239 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=05 | valid_mse=13224.174808 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=06 | valid_mse=13221.827750 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=07 | valid_mse=13245.218443 | lr=2.50e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
Early stopping (patience=6). Best valid_mse=13180.107791

--- GRU GRID 2/6 | h=90 | cfg={'hidden_size': 128, 'lr': 0.0005, 'weight_decay': 0.0001} ---
epoch=01 | valid_mse=13184.294171 | lr=5.00e-04 | hs=128 | wd=1.0e-04 | L=1 | do=0.00
epoch=02 | valid_mse=13186.567214 | lr=5.00e-04 | hs=128 | wd=1.0e-0

In [43]:
df_gru_grid_all = pd.concat([df_gru_grid_60, df_gru_grid_90], ignore_index=True)
df_gru_grid_all

,model,split,horizon_min,MAE,RMSE,R2,DA,hidden_size,lr,weight_decay,best_valid_mse,epochs_ran,final_lr
0,gru,test,60,39.966123,54.355875,-0.000319,0.473416,128,0.0002,0.0001,8641.232312,7,0.000050
1,gru,test,60,40.107339,54.379298,-0.001181,0.477618,256,0.0010,0.0001,8651.471551,7,0.000250
2,gru,test,60,40.030757,54.380716,-0.001233,0.463592,256,0.0002,0.0001,8643.034023,7,0.000050
3,gru,test,60,40.099766,54.400381,-0.001957,0.465863,128,0.0005,0.0001,8640.623880,7,0.000125
4,gru,test,60,40.157535,54.421316,-0.002729,0.465792,128,0.0010,0.0001,8646.281690,7,0.000250
5,gru,test,60,40.506521,54.637781,-0.010721,0.465182,256,0.0005,0.0001,8665.830914,7,0.000125
6,gru,valid,60,61.654870,92.954957,0.000420,0.478532,128,0.0005,0.0001,8640.623880,7,0.000125
7,gru,valid,60,61.536423,92.958229,0.000350,0.477982,128,0.0002,0.0001,8641.232312,7,0.000050
8,gru,valid,60,61.594547,92.967921,0.000142,0.477360,256,0.0002,0.0001,8643.034023,7,0.000050
9,gru,valid,60,61.720842,92.985384,-0.000234,0.477149,128,0.0010,0.0001,8646.281690,7,0.000250


### **7.11. Selección de las mejores filas con VALID**


In [46]:
import pandas as pd

def select_best_valid_then_match_test(
    df: pd.DataFrame,
    *,
    model_name: str = "gru",
    primary_metric: str = "RMSE",
) -> pd.DataFrame:
    """
    Para cada horizon_min:
      1) selecciona la mejor fila en VALID según primary_metric (menor es mejor),
         con desempates: MAE (menor), R2 (mayor), DA (mayor)
      2) busca la fila TEST que tenga EXACTAMENTE los mismos hiperparámetros
         (hidden_size, lr, weight_decay) y el mismo horizon_min.
    Devuelve un DF con 4 filas: valid60, test60, valid90, test90.
    """
    # columnas que definen el "mismo modelo entrenado"
    hp_cols = ["hidden_size", "lr", "weight_decay"]

    df_ = df.copy()

    # Asegurar tipos comparables (evita mismatches por floats)
    for c in hp_cols:
        df_[c] = df_[c].astype(float)

    # ---- 1) Mejor VALID por horizonte ----
    df_valid = df_[df_["split"] == "valid"].copy()

    # Orden: RMSE asc, MAE asc, R2 desc, DA desc
    df_valid = df_valid.sort_values(
        by=[primary_metric, "MAE", "R2", "DA"],
        ascending=[True, True, False, False],
    )

    best_valid = df_valid.groupby("horizon_min", as_index=False).head(1)

    # ---- 2) Matchear TEST por mismos hiperparámetros ----
    df_test = df_[df_["split"] == "test"].copy()

    matched_rows = []
    for _, row in best_valid.iterrows():
        h = int(row["horizon_min"])
        hs, lr, wd = float(row["hidden_size"]), float(row["lr"]), float(row["weight_decay"])

        # match exacto (con tolerancia por float)
        cand = df_test[
            (df_test["horizon_min"] == h)
            & (df_test["hidden_size"].astype(float) == hs)
            & (df_test["lr"].astype(float) == lr)
            & (df_test["weight_decay"].astype(float) == wd)
        ]

        if cand.empty:
            # fallback robusto: merge con rounding (por si lr viene como 0.0005 vs 5e-4)
            df_test_tmp = df_test.copy()
            df_test_tmp["lr_r"] = df_test_tmp["lr"].astype(float).round(10)
            df_test_tmp["wd_r"] = df_test_tmp["weight_decay"].astype(float).round(10)
            df_test_tmp["hs_r"] = df_test_tmp["hidden_size"].astype(float).round(10)

            lr_r = round(lr, 10); wd_r = round(wd, 10); hs_r = round(hs, 10)

            cand = df_test_tmp[
                (df_test_tmp["horizon_min"] == h)
                & (df_test_tmp["hs_r"] == hs_r)
                & (df_test_tmp["lr_r"] == lr_r)
                & (df_test_tmp["wd_r"] == wd_r)
            ].drop(columns=["lr_r", "wd_r", "hs_r"])

        if cand.empty:
            raise ValueError(
                f"No se encontró fila TEST para h={h} con hparams: "
                f"hidden_size={hs}, lr={lr}, weight_decay={wd}. "
                "Revise si esos hparams existen en TEST."
            )

        # si hubiera más de una, tomamos la primera (debería ser única)
        matched_rows.append(cand.iloc[0])

    best_test = pd.DataFrame(matched_rows)

    # ---- salida final: valid+test por horizonte ----
    out = pd.concat([best_valid, best_test], ignore_index=True)

    # Orden final: valid 60/90, test 60/90
    split_order = pd.Categorical(out["split"], categories=["valid", "test"], ordered=True)
    out = out.assign(split=split_order).sort_values(["split", "horizon_min"]).reset_index(drop=True)

    # columnas a mostrar (ajuste si quiere)
    cols = [
        "split","horizon_min","MAE","RMSE","R2","DA",
        "hidden_size","lr","weight_decay","best_valid_mse","epochs_ran","final_lr"
    ]
    return out[cols]

df_gru_selected = select_best_valid_then_match_test(df_gru_grid_all, primary_metric="RMSE")
df_gru_selected


,split,horizon_min,MAE,RMSE,R2,DA,hidden_size,lr,weight_decay,best_valid_mse,epochs_ran,final_lr
0,valid,60,61.654870,92.954957,0.000420,0.478532,128.0,0.0005,0.0001,8640.623880,7,0.000125
1,valid,90,75.830386,114.804650,0.000580,0.476611,128.0,0.0010,0.0001,13180.107791,7,0.000250
2,test,60,40.099766,54.400381,-0.001957,0.465863,128.0,0.0005,0.0001,8640.623880,7,0.000125
3,test,90,48.935671,67.311733,-0.003903,0.457576,128.0,0.0010,0.0001,13180.107791,7,0.000250


## **8. Métricas ML**

In [50]:
cols = ["split", "horizon_min", "MAE", "RMSE", "R2", "DA"]

df_gru_metrics_tuned = df_gru_selected[cols].copy()

# Agregar columna 'model' al inicio con valor constante 'gru'
df_gru_metrics_tuned.insert(0, "model", "gru")

In [51]:
df_gru_metrics_tuned

,model,split,horizon_min,MAE,RMSE,R2,DA
0,gru,valid,60,61.654870,92.954957,0.000420,0.478532
1,gru,valid,90,75.830386,114.804650,0.000580,0.476611
2,gru,test,60,40.099766,54.400381,-0.001957,0.465863
3,gru,test,90,48.935671,67.311733,-0.003903,0.457576


**LSTM tuning**


| # | model | split | horizon_min | MAE       | RMSE       | R2        | DA       |
|---|-------|-------|-------------|-----------|------------|-----------|----------|
| 0 | lstm  | test  | 60          | 40.062974 | 54.359750  | -0.000461 | 0.475432 |
| 1 | lstm  | valid | 60          | 61.611558 | 92.909564  | 0.001396  | 0.484279 |
| 2 | lstm  | test  | 90          | 48.837803 | 67.269610  | -0.002647 | 0.463057 |
| 3 | lstm  | valid | 90          | 75.718288 | 114.708905 | 0.002246  | 0.484445 |


## **9. Guardar artefactos para Stage_08**

In [53]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"metrics_{name}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [54]:
save_seq2one_metrics(
    df_gru_grid_all,
    name="gru_grid_tuning")

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_gru_grid_tuning.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_gru_grid_tuning.parquet')

In [55]:
save_seq2one_metrics(
    df_gru_metrics_tuned,
    name="gru_tuned",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_gru_tuned.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_gru_tuned.parquet')

## **10. Resultados y conclusiones parciales**

**Conclusiones parciales – Modelos recurrentes tuneados (LSTM vs GRU)**

A continuación se presenta una comparación directa entre los modelos **LSTM** y **GRU** en configuración **seq2one**, utilizando los mejores hiperparámetros seleccionados en **VALID** y evaluados posteriormente en **TEST**.

---

**Horizonte 60 minutos**

- **Error (MAE / RMSE):**
  - LSTM y GRU muestran desempeños **muy similares**, sin diferencias estadísticamente relevantes.
- **Directional Accuracy (DA):**
  - **LSTM** presenta un valor ligeramente superior.
- **Conclusión:**
  - Ambos modelos capturan la magnitud del movimiento de forma comparable,  
    pero **LSTM conserva mejor la información direccional**.

---

**Horizonte 90 minutos**

- **Error (MAE / RMSE):**
  - Nuevamente, **no se observan mejoras claras** entre LSTM y GRU.
- **Directional Accuracy (DA):**
  - **LSTM mantiene una ventaja consistente** frente a GRU.
- **Conclusión:**
  - A horizontes más largos, **LSTM preserva mejor la señal direccional**,  
    mientras que GRU pierde estabilidad.

---

**Lectura global**

- Ninguno de los dos modelos logra **reducciones significativas de MAE o RMSE** respecto a modelos más simples (Lasso, MLP).
- **LSTM supera sistemáticamente a GRU en Directional Accuracy**, métrica más relevante desde el punto de vista operativo.
- La menor complejidad del GRU **no se traduce en mejor generalización**.

---

**Decisión operativa sugerida**

- **Conservar LSTM** como modelo recurre